# Project Phase 3 Orchestrator Wrapper

Notebook wrapper for the `prototype` agentic application for Peloton Fitness

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "prototype").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "Project_Phase_3" / "prototype").exists():
    sys.path.insert(0, str(cwd / "Project_Phase_3"))

from prototype.orchestrator import AgenticOrchestrator

orchestrator = AgenticOrchestrator()
print("Orchestrator initialized.")


## Declare "ask" function to interact with the graph

In [ ]:
import json

def _render_chart_from_story_output(story_output):
    if not isinstance(story_output, dict):
        return False

    chart_spec = story_output.get("chart_spec")
    if not isinstance(chart_spec, dict):
        return False

    if chart_spec.get("library") != "plotly":
        print("[chart] Unsupported chart library in story_output.chart_spec")
        return False

    try:
        import plotly.graph_objects as go
    except Exception:
        print("[chart] Plotly is not installed; cannot render chart_spec.")
        return False

    data = chart_spec.get("data", [])
    layout = chart_spec.get("layout", {})
    try:
        fig = go.Figure(data=data, layout=layout)
        fig.show()
        return True
    except Exception as exc:
        print(f"[chart] Failed to render chart_spec: {exc}")
        return False


def ask(
    user_query: str,
    thread_id: str = "default",
    show_story_output: bool = False,
    debug: bool = False,
    render_chart: bool = True,
):
    out = orchestrator.invoke(user_query, thread_id=thread_id)
    print("USER:", user_query)
    print("THREAD:", thread_id)
    print("\nASSISTANT:")
    print(out["response"])
    print(f"\n[routed domain={out['active_domain']} story={out['active_story_id']}]")

    if debug:
        rm = out.get("router_metrics", {})
        print("\nrouter_reason:", out.get("router_reason"))
        print("continuation_score:", rm.get("continuation_score"))
        print("domain_scores:", rm.get("domain_scores"))
        print("margin:", rm.get("margin"))
        print("domain_selected_by:", rm.get("domain_selected_by"))
        print("story_selected_by:", rm.get("story_selected_by"))
        print("domain_guardrail_override:", rm.get("domain_guardrail_override"))

        so = out.get("story_output") or {}
        if isinstance(so, dict):
            ep = so.get("evidence_plan") or {}
            lr = so.get("llm_rationale") or {}
            if ep or lr:
                print("\nllm_story_debug:")
                print("planner_source:", ep.get("planner_source"))
                print("planner_confidence:", ep.get("planner_confidence"))
                print("llm_rationale_source:", lr.get("source"))
                print("llm_rationale_confidence:", lr.get("confidence"))

            plan_snapshot = so.get("plan_snapshot") or {}
            if so.get("planner_source") or plan_snapshot:
                print("\nplanning_debug:")
                print("planner_source:", so.get("planner_source"))
                print("planner_confidence:", plan_snapshot.get("planner_confidence"))
                print("needs_clarification:", so.get("needs_clarification", plan_snapshot.get("needs_clarification")))
                print("requested_slot:", so.get("requested_slot", plan_snapshot.get("requested_slot")))

    if render_chart:
        _render_chart_from_story_output(out.get("story_output"))

    if show_story_output and out.get("story_output") is not None:
        print("\nstory_output:")
        try:
            print(json.dumps(out["story_output"], indent=2, default=str))
        except Exception:
            print(out["story_output"])

    return out





## Orchestrator Graph (LangGraph + Mermaid)
Top-level orchestrator graph visualization

In [ ]:
from IPython.display import Markdown, display, Image

mermaid = orchestrator.get_orchestrator_mermaid()
display(Markdown("```mermaid\n" + mermaid + "\n```"))

try:
    png = orchestrator.get_orchestrator_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


## Membership Fraud Story Graph
Render the LangGraph for `mf_story_1` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.membership_fraud_story1 import get_membership_fraud_story1_mermaid

mf_mermaid = get_membership_fraud_story1_mermaid()
display(Markdown("```mermaid\n" + mf_mermaid + "\n```"))

try:
    # Build an ephemeral graph image from the same definition used by the story module
    from prototype.stories.membership_fraud_story1 import _get_security_graph
    png = _get_security_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "mf_security_check"

print("1) Missing member_id -> should ask for member_id")
_ = ask("Can you check my most recent suspicious login?", thread_id=thread_id, debug=True)

print("\n2) Provide member_id -> should retrieve alert(s)")
_ = ask("MB001", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Not recognized -> should escalate actions")
_ = ask("I do not recognize that login.", thread_id=thread_id, debug=True)

#print("\n4) Recognized -> should reassure")
#_ = ask("Yes that was me.", thread_id=thread_id, debug=True)

print("\n5) Follow-up how-to: password change (should use security help KB)")
_ = ask("I do not recognize it. How do I change my password?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Follow-up how-to: MFA setup (should use security help KB)")
_ = ask("How do I set up multi-factor authentication?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n7) Follow-up how-to: sign out sessions (should use security help KB)")
_ = ask("How do I sign out of all active sessions?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n8) Non-howto check: still recognized flow")
_ = ask("I recognize this device now.", thread_id=thread_id, debug=True)


In [ ]:
thread_id = "mf_ambiguous_issue"

print("\n4) Ambiguous issue -> should require human review")
_ = ask("Something is wrong with my account and I need help.", thread_id=thread_id, debug=True, show_story_output=True)

In [ ]:
thread_id = "mf_final_check"
print("\n6) This question should route to mf_story_2 and should be high confidence RAG, but low-relevance KB results should cause it to route to human instead")
_ = ask("My bill has a charge that I know is incorrect.", thread_id=thread_id, debug=True, show_story_output=True)

In [ ]:
thread_id = "mf_issue_triage_demo"

print("1) Login issue triage + high-confidence RAG")
_ = ask("I cannot login and need a password reset because I am locked out.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Billing issue triage + queue routing")
_ = ask("I was charged twice and need a refund on my latest invoice.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Renewal issue triage + queue routing")
_ = ask("My subscription renewal failed and my membership expired.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Ambiguous issue -> should require human review")
_ = ask("Something is wrong with my account and I need help.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Security-alert phrasing should route to mf_story_1 (not mf_story_2)")
_ = ask("I got a suspicious login alert from a new device in another location.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) This question should route to mf_story_2 and should be high confidence RAG, but low-relevance KB results should cause it to route to human instead")
_ = ask("Billing issue: my invoice shows a NACHA return code R29 and a network-token lifecycle mismatch; I was charged once, but the gateway flagged a soft-descriptor anomaly.", thread_id=thread_id, debug=True, show_story_output=True)

In [ ]:
thread_id = "mf_tier_fit_demo"

print("1) Missing member_id -> should ask for member_id")
_ = ask("Can you analyze whether my membership tier is optimal?", thread_id=thread_id, debug=True)

print("\n2) Provide member_id + timeframe -> should route to mf_story_3 and evaluate tier fit")
_ = ask("MB017 over the last 6 months", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Ask for benefit-sensitive recommendation -> should use evidence planner and feature breakdown")
_ = ask("Please include benefits usage context and explain upgrade/downgrade options.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Follow-up what-if style phrasing in same thread")
_ = ask("If my usage stays similar, should I keep this tier?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Upgrade scenario example (expected decision=upgrade)")
_ = ask("My member ID is MB010. Based on usageover the last 6 months, is my membership tier still right?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Downgrade scenario example (expected decision=downgrade)")
_ = ask("My member ID is MB034. Based on my usage over the recent monthly average, should I downgrade my membership tier?", thread_id=thread_id, debug=True, show_story_output=True)


## Business Marketing Story Graphs
Render the LangGraph for `bm_story_1`, `bm_story_2`, and `bm_story_3`.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.business_marketing_story1 import get_business_marketing_story1_mermaid, _get_business_marketing_graph

bm1_mermaid = get_business_marketing_story1_mermaid()
display(Markdown("```mermaid\n" + bm1_mermaid + "\n```"))

try:
    png = _get_business_marketing_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.business_marketing_story2 import get_business_marketing_story2_mermaid, _get_business_marketing_story2_graph

bm2_mermaid = get_business_marketing_story2_mermaid()
display(Markdown("```mermaid\n" + bm2_mermaid + "\n```"))

try:
    png = _get_business_marketing_story2_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.business_marketing_story3 import get_business_marketing_story3_mermaid, _get_business_marketing_story3_graph

bm3_mermaid = get_business_marketing_story3_mermaid()
display(Markdown("```mermaid\n" + bm3_mermaid + "\n```"))

try:
    png = _get_business_marketing_story3_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "bm_feedback_check"

print("1) Campaign + channels + timeframe")
_ = ask("Summarize last month feedback for CAMP105 across app and email and suggest adjustments", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Widened request (no campaign filter)")
_ = ask("Summarize last 8 weeks of marketing feedback and suggest 3 content adjustments", thread_id=thread_id, debug=True)

print("\n3) Sparse/no-match request")
_ = ask("Summarize last week feedback for CAMP999 across social", thread_id=thread_id, debug=True)


In [ ]:
thread_id = "bm_kpi_check"

print("1) Underperformers-only intent (threshold-focused)")
_ = ask("Find underperforming weekly campaign metrics by channel and target segment for last month using CTR, CAC, and ROAS thresholds", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Compare intent + metric scoping (ROAS/CAC only)")
_ = ask("Compare ROAS and CAC by campaign for last quarter and include trend deltas versus prior weeks", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Overview intent (full KPI summary)")
_ = ask("Give me a weekly KPI summary for last month by channel and target segment", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Definitions intent")
_ = ask("What are CTR, CAC, and ROAS?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Concise underperformers request")
_ = ask("Only show underperformers for last month by channel. Keep it brief.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Unsupported-dimension guardrail check")
_ = ask("Show weekly campaign performance by geography for last month", thread_id=thread_id, debug=True, show_story_output=True)


In [ ]:
thread_id = "bm_leads_check"

print("1) Default assumptions (recent + email) with top-N")
_ = ask("Generate prioritized leads and draft follow-up messages for top 5", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Explicit timeframe + channel + interest filter")
_ = ask("For last 14 days, generate top 6 cycling leads and draft call follow-ups in a friendly tone", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Suppression guardrail check (email channel)")
_ = ask("Show top 12 leads for last 7 days and draft email outreach", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Unsupported lookback coercion check")
_ = ask("Give me top 8 leads for last 10 days and draft email follow-up messages", thread_id=thread_id, debug=True, show_story_output=True)


In [ ]:
# Cell A: preflight check
import os
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing in this kernel environment.")
print("OPENAI_API_KEY detected.")


In [ ]:
# Cell B: Story 3 orchestrator smoke + routing checks
thread_id = "bm_leads_check"

cases = [
    ("defaults", "Generate prioritized leads and draft follow-up messages for top 5"),
    ("explicit", "For last 14 days, generate top 6 cycling leads and draft call follow-ups in a friendly tone"),
    ("suppression_email", "Show top 12 leads for last 7 days and draft email outreach"),
    ("coercion", "Give me top 8 leads for last 10 days and draft email follow-up messages"),
    ("empty_query", "   "),
]

results = {}
for label, q in cases:
    print(f"\n--- CASE: {label} ---")
    out = ask(q, thread_id=thread_id, debug=True, show_story_output=True)
    results[label] = out

    assert out["active_domain"] == "business_marketing", f"{label}: wrong domain"
    assert out["active_story_id"] == "bm_story_3", f"{label}: wrong story"

print("\nAll routing checks passed for bm_story_3.")


In [ ]:
# Cell C: contract/behavior assertions specific to Story 3
d = results["defaults"]["story_output"]
assert d["lookback_days"] == 7
assert d["channel"] == "email"
assert any("defaulted to last 7 days" in x.lower() for x in d.get("assumptions", []))
assert any("defaulted to email" in x.lower() for x in d.get("assumptions", []))

c = results["coercion"]["story_output"]
assert c["lookback_days"] == 7
assert any("mapped to supported 7-day lookback" in x.lower() or "normalized to 7 days" in x.lower() for x in c.get("assumptions", []))

e = results["suppression_email"]["story_output"]
ranked_ids = {r.get("lead_id") for r in e.get("ranked_leads", [])}
assert "L003" not in ranked_ids, "Expected email suppression to remove L003"
assert int(e.get("suppressed_excluded_count", 0)) >= 1

# Important orchestrator integration check: no member-id pending slot side effects from Story 3
state = results["empty_query"]["state"]
assert state.get("pending_slot_type") in {None, "member_id"}  # should remain unchanged by Story 3 clarification
print("Story 3 behavior checks passed.")


In [ ]:
# Cell D: Confidence-path checks for bm_story_3 (high infer / mid clarify / low default)
thread_id = "bm_story3_confidence_paths"

tests = [
    # likely explicit controls
    ("explicit_baseline", "For last 14 days, generate top 6 cycling leads and draft call follow-ups in a friendly tone"),
    # likely high-confidence inference candidate (timeframe implied, channel unspecified)
    ("infer_candidate", "Prioritize recent high-intent leads and draft outreach for top 8"),
    # likely mid-confidence ambiguity to trigger clarification
    ("clarify_candidate", "Generate top leads and draft follow-ups"),
    # low-information prompt to force defaults/clarify behavior
    ("low_info", "help with leads"),
]

outs = {}
for label, q in tests:
    print(f"\n=== {label} ===")
    out = ask(q, thread_id=thread_id, debug=True, show_story_output=True)
    outs[label] = out

    so = out.get("story_output", {}) or {}
    fr = so.get("field_resolution", {})
    print("\nfield_resolution:")
    print(fr)
    print("unresolved_fields:", so.get("unresolved_fields"))
    print("clarification_question:", so.get("clarification_question"))


In [ ]:
# Cell E: Soft assertions (robust to LLM variability, strict on contract shape)
for label, out in outs.items():
    assert out["active_domain"] == "business_marketing", f"{label}: wrong domain"
    assert out["active_story_id"] == "bm_story_3", f"{label}: wrong story"

    so = out.get("story_output", {}) or {}

    # If clarification path, contract should include these keys
    if so.get("needs_request_details"):
        assert "field_resolution" in so, f"{label}: missing field_resolution on clarify path"
        assert "unresolved_fields" in so, f"{label}: missing unresolved_fields on clarify path"
        assert isinstance(so.get("unresolved_fields"), list), f"{label}: unresolved_fields must be list"
        continue

    # Non-clarify path contract
    for k in ["lookback_days", "channel", "top_n", "assumptions", "field_resolution"]:
        assert k in so, f"{label}: missing {k}"

    assert so["lookback_days"] in {7, 14, 30}, f"{label}: invalid lookback_days"
    assert so["channel"] in {"email", "call", "sms"}, f"{label}: invalid channel"
    assert 1 <= int(so["top_n"]) <= 100, f"{label}: invalid top_n"

    fr = so["field_resolution"]
    for fk in ["lookback_days", "channel", "primary_class_interest"]:
        assert fk in fr, f"{label}: missing field_resolution.{fk}"
        assert "source" in fr[fk], f"{label}: missing source for {fk}"
        assert "confidence" in fr[fk], f"{label}: missing confidence for {fk}"

print("Contract checks passed.")


In [ ]:
# Cell F: Optional stricter checks for specific expectations
# Note: these may fail occasionally with model variance; keep optional.
explicit = outs["explicit_baseline"]["story_output"]
assert explicit.get("lookback_days") == 14
assert explicit.get("channel") == "call"

print("Optional explicit checks passed.")


In [ ]:
# Cell G: Multi-turn ambiguity resolution check in one thread
thread_id = "bm_story3_multiturn_resolution"

print("Turn 1: ambiguous")
t1 = ask("Generate top leads and draft follow-ups", thread_id=thread_id, debug=True, show_story_output=True)

print("\nTurn 2: resolve ambiguity explicitly")
t2 = ask("Use last 14 days and email channel, focus on cycling", thread_id=thread_id, debug=True, show_story_output=True)

print("\nTurn 3: refine only tone/top_n")
t3 = ask("Make tone consultative and show top 5", thread_id=thread_id, debug=True, show_story_output=True)

# Expected: stays in business_marketing / bm_story_3 across turns
for i, t in enumerate([t1, t2, t3], start=1):
    assert t["active_domain"] == "business_marketing", f"turn {i}: wrong domain"
    assert t["active_story_id"] == "bm_story_3", f"turn {i}: wrong story"

print("Multi-turn routing checks passed.")


## Data Science Story Graphs
Render the LangGraph for `ds_story_2` and `ds_story_3`, and run demonstration checks including the new viz story `ds_story_1`.


In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.data_science_story2 import get_data_science_story2_mermaid, _get_data_science_graph

ds_mermaid = get_data_science_story2_mermaid()
display(Markdown("```mermaid\n" + ds_mermaid + "\n```"))

try:
    png = _get_data_science_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.data_science_story3 import get_data_science_story3_mermaid, _get_peer_benchmark_graph

ds3_mermaid = get_data_science_story3_mermaid()
display(Markdown("```mermaid\n" + ds3_mermaid + "\n```"))

try:
    png = _get_peer_benchmark_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


## Data Science Viz Story Demo (`ds_story_1`)
Run these to test chart-building behavior and auto-rendered Plotly charts from `story_output.chart_spec`.


In [ ]:
thread_id = "ds_viz_story1_check"

print("1) Member-scoped trend line chart")
_ = ask("For MB001, create a weekly trend chart of duration over the last 8 weeks.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Bar chart by workout type")
_ = ask("For MB001, show a bar chart of average calories by workout type over the last 8 weeks.", thread_id=thread_id, debug=True)

print("\n3) Scatter chart for metric relationship")
_ = ask("For MB001, create a scatter plot of output_kj versus calories in the last 8 weeks.", thread_id=thread_id, debug=True)

print("\n4) Histogram distribution chart")
_ = ask("For MB001, plot a histogram of strive score for the last 8 weeks.", thread_id=thread_id, debug=True)

print("\n5) Box plot by class type")
_ = ask("For MB001, make a box plot of duration by workout type over the last 8 weeks.", thread_id=thread_id, debug=True)


In [ ]:
thread_id = "ambiguous_visuals_check"

print("1) Ambiguous chart request -> should develop a plan of possible visualizations and select the best one.")
_ = ask("Show me a chart of my progress for MB001", thread_id="ds_viz_story1_check", show_story_output=True, debug=True)

print("\n2) Ambiguous chart request -> should develop a plan of possible visualizations and select the best one.")
_ = ask("Visualize output and calories for MB001", thread_id="ds_viz_story1_check", show_story_output=True, debug=True)

## Data Science Viz Refinement Loop Demo
Use one thread to validate carry-forward refinement in `ds_story_1` (deterministic refinement intent detection).


In [ ]:
thread_id = "ds_viz_refinement_check_"

print("1) Initial ambiguous request")
_ = ask("Show me a chart of my progress for MB001", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Refinement: switch chart type and metric relation")
_ = ask("Switch to scatter with calories vs output_kj", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Refinement: keep same chart, change timeframe")
_ = ask("Keep the same chart but use the last 12 weeks", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Refinement: switch segmentation")
_ = ask("Now segment by weekday instead", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Provides clarification of domain")
_ = ask("Data science: Can we look at the same information grouped by weekday?", thread_id=thread_id, debug=True, show_story_output=True)

In [ ]:
thread_id = "ds_workout_trends_check"

print("1) Missing member_id -> should ask for member_id")
_ = ask("Am I improving over the past 8 weeks?", thread_id=thread_id, debug=True)

print("\n2) Provide member + trend question")
_ = ask("For MB001, am I improving in my workouts over the last 8 weeks?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Drivers + intensity question")
_ = ask("For MB001 what is driving intensity and do weekdays differ?", thread_id=thread_id, debug=True)

print("\n4) Anomaly question")
_ = ask("For MB001, any unusual drops or spikes in the last 8 weeks?", thread_id=thread_id, debug=True)


In [ ]:
thread_id = "ds_peer_benchmark_check"

print("1) Generic metrics request -> should ask which metrics")
_ = ask("Compare MB001 metrics to peers.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Clarify metrics -> should proceed with selected metrics")
_ = ask("Use weekly workouts and consistency.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Ask for custom peer definition")
_ = ask("Now compare me to peers with similar activity level over the last 12 weeks.", thread_id=thread_id, debug=True, show_story_output=True)

In [ ]:
thread_id = "ds_peer_benchmark_high_uncertainty"

print("1) Vague cohort + vague metrics")
_ = ask("Can you benchmark me against people like me and tell me what to fix? My member ID is MB001.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Generic metrics wording")
_ = ask("How am I doing versus peers? Use the right metrics.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Underspecified cohort constraints")
_ = ask("Compare me to others but not everyone, and focus on what matters most.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Ambiguous but intentful peer comparison")
_ = ask("I want a fair peer comparison for my workouts and advice from that.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Similar-user phrasing + vague timeframe")
_ = ask("Show where I stand vs similar users recently.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Broad request without slots")
_ = ask("Do a peer check for me and recommend next steps.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n7) Hedged metric selection")
_ = ask("Compare my performance with peers, maybe by consistency or something.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n8) Cohort mention without definition")
_ = ask("What should I improve compared to others in my cohort?", thread_id=thread_id, debug=True, show_story_output=True)


## Interactive Multi-Turn Loop
Run this cell for live manual testing. Type `exit` or `quit` to stop.

In [ ]:
active_thread_id = input("Thread ID (default=user_live): " ).strip() or "user_live"
print(f"Interactive multi-turn mode started for thread_id={active_thread_id}. Type 'exit' to stop.")
while True:
    user_text = input("You: " ).strip()
    if user_text.lower() in {"exit", "quit"}:
        print("Stopped interactive mode.")
        break
    if not user_text:
        continue

    out = ask(user_text, thread_id=active_thread_id)
    print(f"(thread={active_thread_id}, domain={out['active_domain']}, story={out['active_story_id']})")
    print('-' * 80)

In [ ]:
active_thread_id = input("Thread ID (default=user_live): " ).strip() or "user_live"
print(f"Interactive multi-turn mode started for thread_id={active_thread_id}. Type 'exit' to stop.")
while True:
    user_text = input("You: " ).strip()
    if user_text.lower() in {"exit", "quit"}:
        print("Stopped interactive mode.")
        break
    if not user_text:
        continue

    out = ask(user_text, thread_id=active_thread_id, debug=True)
    print("pending_slot_type:", out["state"]["pending_slot_type"])
    print("member:", out["state"]["member"])

    print(f"(thread={active_thread_id}, domain={out['active_domain']}, story={out['active_story_id']})")
    print('-' * 80)

## Inspect state

In [ ]:
orchestrator.state